## 코랩 실행

In [ ]:
# 구글드라이브 연동
from google.colab import drive
drive.mount('/gdrive', force_remount=True)

# 구글 드라이브 파일 확인
!ls '/gdrive/My Drive/temp/data'

# 반복되는 드라이브 경로 변수화
drive_path = '/gdrive/My Drive/temp/data/'

Mounted at /gdrive
beauty_buy.csv	buy_w2019.csv  period2.xlsx   sns2018_2.csv  table2019.csv
beauty_sns.csv	food_buy.csv   period3.xlsx   sns_2018.csv   table2019.txt
buy2018_1.csv	food_sns.csv   period4.xlsx   sns2019_1.csv  table2020.csv
buy2018_2.csv	home_buy.csv   period5.xlsx   sns2019_2.csv  weather2018.csv
buy_2018.csv	home_sns.csv   period6.xlsx   sns_2019.csv   weather2019.csv
buy2019_1.csv	period10.xlsx  period7.xlsx   sns_w2018.csv
buy2019_2.csv	period11.xlsx  period8.xlsx   sns_w2019.csv
buy_2019.csv	period12.xlsx  period9.xlsx   table2018.csv
buy_w2018.csv	period1.xlsx   sns2018_1.csv  table2018.txt


## 기본적인 라이브러리 호출

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import datetime
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# 데이터 불러오기
df_buy = pd.read_csv(drive_path + 'buy2018_1.csv')
df_buy2 = pd.read_csv(drive_path + 'buy2018_2.csv')

df_buy3 = pd.read_csv(drive_path + 'buy2019_1.csv')
df_buy4 = pd.read_csv(drive_path + 'buy2019_2.csv')

df_sns = pd.read_csv(drive_path + 'sns2018_1.csv', encoding='cp949')
df_sns2 = pd.read_csv(drive_path + 'sns2018_2.csv', encoding='cp949')

df_sns3 = pd.read_csv(drive_path + 'sns2019_1.csv', encoding='cp949')
df_sns4 = pd.read_csv(drive_path + 'sns2019_2.csv', encoding='cp949')

weather2018 = pd.read_csv(drive_path + 'table2018.txt')
weather2019 = pd.read_csv(drive_path + 'table2019.txt')

In [ ]:
# 열 맞춰주기
buy_col = list(df_buy.columns)
df_buy2.columns = [i for i in buy_col]

buy_col2 = list(df_buy3.columns)
df_buy4.columns = [i for i in buy_col2]

sns_col = list(df_sns)
df_sns2.columns = [i for i in sns_col]

sns_col2 = list(df_sns3)
df_sns4.columns = [i for i in sns_col2]

In [ ]:
# 필요없는거 제거
buy_2018 = pd.concat([df_buy, df_buy2], ignore_index=True)
buy_2018.drop(columns=['Unnamed: 0'], axis=1, inplace=True)

buy_2019 = pd.concat([df_buy3, df_buy4], ignore_index=True)
buy_2019.drop(columns=['Unnamed: 0'], axis=1, inplace=True)

sns_2018 = pd.concat([df_sns, df_sns2], ignore_index = True)
sns_2018.drop(columns=['Column1'], axis=1, inplace=True)

sns_2019 = pd.concat([df_sns3, df_sns4], ignore_index = True)
sns_2019.drop(columns=['Column1'], axis=1, inplace=True)

In [ ]:
buy_2018.columns = ['date', 'sex', 'age', 'big', 'small', 'qty']
buy_2019.columns = ['date', 'sex', 'age', 'big', 'small', 'qty']
sns_2018.columns = ['date', 'big', 'small', 'cnt']
sns_2019.columns = ['date', 'big', 'small', 'cnt']

In [ ]:
buy_2018['date'] = buy_2018['date'].astype('str')
buy_2018['date'] = pd.to_datetime(buy_2018['date'])

buy_2019['date'] = buy_2019['date'].astype('str')
buy_2019['date'] = pd.to_datetime(buy_2019['date'])

sns_2018['date'] = sns_2018['date'].astype('str')
sns_2018['date'] = pd.to_datetime(sns_2018['date'])

sns_2019['date'] = sns_2019['date'].astype('str')
sns_2019['date'] = pd.to_datetime(sns_2019['date'])

In [ ]:
buy = pd.concat([buy_2018, buy_2019], ignore_index=True)
sns = pd.concat([sns_2018, sns_2019], ignore_index=True)

In [ ]:
beauty_buy = buy[buy['big'] == '뷰티'].reset_index(drop=True)
food_buy = buy[buy['big'] == '식품'].reset_index(drop=True)
home_buy = buy[buy['big'] == '냉난방가전'].reset_index(drop=True)

beauty_sns = sns[sns['big'] == '뷰티'].reset_index(drop=True)
food_sns = sns[sns['big'] == '식품'].reset_index(drop=True)
home_sns = sns[sns['big'] == '냉난방가전'].reset_index(drop=True)

## 중분류 선별

In [ ]:
array = {'가공란':'가공식품', '가자미':'수산물', '갈비/찜/바비큐용 돈육':'육류', 
       '갈비용 우육':'육류', '갈치':'수산물', '감/홍시':'과일', '감귤/한라봉/오렌지':'과일',
       '감마리놀렌산 영양제':'건강식품', '감말랭이':'건과', '감자':'채소', '갓김치':'김치',
       '건강즙':'건강식품', '건강즙/녹용':'건강식품', '건대추':'건과', '건망고':'건과',
       '건바나나':'건과', '건어물 건새우':'건어물', '건어물 노가리':'건어물', '건어물 마른오징어':'건어물', 
       '건어물 멸치':'건어물', '건어물 쥐포':'건어물',
       '건어물 진미채':'건어물', '건어물 황태':'건어물', '건자두':'건과', 
       '건포도':'건과', '게장류':'반찬', '견과류':'견과', '견과류 땅콩':'견과',
       '견과류 마카다미아':'견과', '견과류 밤':'견과', '견과류 잣/은행':'견과', 
       '견과류 카카오닙스':'견과', '견과류 캐슈넛':'견과','견과류 피스타치오':'견과', '견과류 호두':'견과', 
       '계란':'알류', '고등어':'수산물', '고추/피망/파프리카':'채소', '곡물차':'음료', '곶감/반건시':'건과',
       '과실차':'음료', '과일류':'과일', '과일세트':'과일', '과채 음료/주스':'음료', 
       '국내산 돈육':'육류', '굴 생물':'수산물', '굴비/조기':'수산물',
       '글루코사민/키토산 영양제':'건강식품', '기타 농산물':'채소', '기타 주스류':'음료', 
       '기타 한방/환제품':'건강식품', '김치류':'김치', '꽃게':'수산물',
       '나물':'채소', '낙지':'수산물', '녹차':'음료', '느타리버섯':'채소', 
       '다이어트보조식':'건강식품', '다이어트용 헬스보충식품':'건강식품', '닭 양념육':'육류',
       '닭가슴살':'육류', '대게/킹크랩':'수산물', '더치커피':'음료', '도라지/더덕':'채소', 
       '돼지 곱창':'육류', '두유':'음료', '둥굴레차':'음료',
       '딸기/복분자/블루베리':'과일', '딸기우유':'음료', '랍스타':'수산물', 
       '레몬/자몽':'과일', '루테인/눈 영양제':'건강식품', '마/야콘':'채소',
       '마늘/생강':'채소', '메추리알':'알류', '명태/동태':'수산물', '무/배추':'채소', 
       '무김치':'김치', '문어':'수산물', '물김치':'김치', '미나리':'채소',
       '미숫가루/곡물가루':'분말가루', '믹스 채소':'채소', '밀크티/티라떼':'음료', 
       '바나나/파인애플/망고':'과일', '바나나우유':'음료', '반건조고구마':'가공식품',
       '반찬류':'반찬', '배/포도/과일즙':'과일', '배추김치':'김치', '백김치':'김치', '보리차':'음료', 
       '복분자/석류/과실즙':'음료', '부추':'채소',
       '브로콜리/셀러리':'채소', '비타민':'건강식품', '비타민/화이바 음료':'음료', 
       '삼치':'수산물', '상황버섯':'채소', '새송이버섯':'채소', '새우/대하':'수산물',
       '생닭/닭부분육':'육류', '생선류':'수산물', '생수':'음료', '생식/선식 분말':'선식류', '선식':'선식류', 
       '소고기 등심/안심':'육류', '소고기 육회':'육류',
       '수산 생물':'수산물', '수입우육':'육류', '숙취/에너지/건강 음료':'음료', 
       '스피루리나 영양제':'건강식품', '시금치':'채소', '식혜/수정과':'음료', '쌀':'곡류',
       '쌈채소':'채소', '아몬드유/코코넛밀크':'음료', '아이스티':'음료', '야채/호박즙':'건강식품', 
       '양념 돈육':'육류', '양념우육':'육류', '양배추/양상추':'채소',
       '양파/마늘즙':'건강식품', '어란(생선알)':'수산물', '어린이 음료':'음료', '어린이영양제':'건강식품', 
       '어린이홍삼':'건강식품', '어린잎/새싹채소':'채소',
       '에이드':'음료', '연어/훈제연어':'수산물', '영지버섯':'채소', '오리고기/훈제오리':'육류', 
       '오메가3/스쿠알렌 영양제':'건강식품', '오이/가지':'채소',
       '오징어':'수산물', '옥돔':'수산물', '옥수수':'채소', '옥수수차':'음료', '옻/칡/쑥즙':'건강식품', 
       '요거트/발효유':'가공식품', '우엉/연근':'채소', '원두커피':'홈카페',
       '윙봉/닭다리/날개':'육류', '유자차':'음료', '유제품 음료':'음료', '율무차':'음료', 
       '음용 식초':'음료', '이온음료':'음료', '인삼/수삼/산삼':'채소',
       '인스턴트커피':'홈카페', '잡곡':'곡류', '잡곡 씨드류':'곡류', '장어':'수산물', 
       '장조림/카레용 돈육':'육류', '저/무지방우유':'음료', '전복 생물':'수산물',
       '전통주':'음료', '전통차':'음료', '절임배추/김치속':'김치', '젓갈류':'반찬', '조개':'수산물', 
       '주꾸미':'수산물', '차 선물세트':'음료', '차/곡물 음료':'음료',
       '참외/메론/수박':'과일', '초유 영양제':'건강식품', '초코우유':'음료', 
       '카페 푸드':'홈카페', '카페용 초콜릿시럽':'홈카페', '칼슘/철분 영양제':'건강식품',
       '캡슐/POD커피':'홈카페', '커피용 프림':'홈카페', '커피음료':'음료', 
       '코코아/핫초코':'음료', '콜라겐/코큐텐 영양제':'건강식품', '콩나물/숙주':'채소',
       '클로렐라 영양제':'건강식품', '키위/참다래':'과일', '탄산수':'음료', 
       '탄산음료':'음료', '토마토':'채소', '파/양파':'채소', '파김치':'김치',
       '포도/거봉/체리':'과일', '표고버섯':'채소', '프라페/버블티 파우더':'분말가루', 
       '프로바이오틱스':'건강식품', '프로폴리스/로얄젤리':'건강식품',
       '한방 분말/환제품':'건강식품', '한방재료':'건강식품', '한우육':'육류', 
       '해조류 다시마':'수산물', '해조류 미역':'수산물', '해초류 ':'수산물', '허브차':'음료',
       '헛개/가시오가피':'건강식품', '호박':'채소', '혼합견과':'견과', 
       '홍삼 간식':'건강식품', '홍삼 분말/환':'건강식품', 
       '홍삼 음료':'건강식품', '홍삼/인삼 제품':'건강식품',
       '홍삼액/홍삼정':'건강식품', '홍삼절편/홍삼정과':'건강식품', '홍어':'수산물', 
       '홍차':'음료', '환자식':'선식류', '회':'수산물', '흰우유':'음료', '구이/수육용 돈육':'육류',
       '혼합곡':'곡류', '과일채소 분말/분태':'분말가루'}

In [ ]:
array2 = {'온풍기':'난방가전', '스탠드형 냉온풍기':'냉방가전', '스탠드형 냉온풍기':'난방가전', '제습기':'기타', '컨벡터':'난방가전', '라디에이터':'난방가전', '이동형 에어컨':'냉방가전', '공기청정기':'공기청정가전',
       '히터':'난방가전', '전기온수기':'기타', '업소용 선풍기':'냉방가전', '멀티형 에어컨':'냉방가전', '에어워셔':'가습가전', '돈풍기':'난방가전', '탁상/USB 선풍기':'냉방가전',
       '스탠드형 에어컨':'냉방가전', '벽걸이 에어컨':'냉방가전', '초음파식 가습기':'가습가전', '휴대용 선풍기':'냉방가전', '온수매트':'난방가전', '벽걸이형 냉온풍기':'냉방가전', '벽걸이형 냉온풍기':'난방가전',
       '보일러':'난방가전', '냉풍기':'냉방가전', '벽걸이형 선풍기':'냉방가전', '자연식 가습기':'가습가전', '중대형 에어컨':'냉방가전', '난방용 열풍기':'난방가전',
       '가열식 가습기':'가습가전', '천장형 에어컨':'냉방가전', '전기장판':'난방가전', '카페트매트':'기타', '공기정화 용품':'공기청정가전', '온열매트':'난방가전',
       '에어컨 리모컨':'냉방가전', '황토매트':'난방가전', '복합식 가습기':'가습가전', '가스온수기':'기타', '산림욕기':'기타', '에어커튼':'냉방가전', '에어커튼':'난방가전', '신발건조기':'기타',
       '의류건조기':'기타'}

In [ ]:
array_beauty = {'기능성 링클케어 화장품':'기능성 화장품', '기능성 모공관리 화장품':'기능성 화장품', 
                '기능성 아이케어 화장품':'기능성 화장품', '기능성 영양보습 화장품':'기능성 화장품',
       '기능성 트러블케어 화장품':'기능성 화장품', '기능성 화장품 세트':'기능성 화장품', 
       '기초 화장용 로션':'스킨케어', '기초 화장용 미스트':'스킨케어',
       '기초 화장용 스킨':'스킨케어', '기초 화장용 에센스':'스킨케어', 
       '기초 화장용 오일/앰플':'스킨케어', '기초 화장용 크림':'스킨케어', '남성 로션':'남성 화장품',
       '남성 메이크업':'남성 화장품', '남성 선케어':'남성 화장품', '남성 세트':'남성 화장품', '남성 쉐이빙':'남성 화장품', 
       '남성 스킨':'남성 화장품', '남성 에센스':'남성 화장품', '남성 크림':'남성 화장품',
       '남성 클렌징':'남성 화장품', '네일 메이크업 용품':'네일', '네일관리 소품':'네일', 
       '네일리무버':'네일', '네일세트':'네일', '네일컬러':'네일', '네일케어':'네일',
       '데오드란트':'바디', '린스':'헤어', '립앤아이 리무버':'클렌징', '메이크업 박스':'뷰티소품', 
       '메이크업 브러쉬':'뷰티소품', '미용가위':'뷰티소품', '바디 보습제':'바디',
       '바디 세트':'바디', '바디 스크럽':'바디', '바디 클렌져':'바디', '바디케어용 땀패드':'바디', '바디케어용 때비누':'바디',
       '바디케어용 볼륨업크림':'바디', '바디케어용 슬리밍':'바디', '바디케어용 제모제':'바디', '바디케어용 청 결제':'바디',
       '베이스 메이크업 세트':'메이크업', '베이스 메이크업용 BB크림':'메이크업', '베이스 메이크업용 CC크림':'메이크업',
       '베이스 메이크업용 가루파우더':'메이크업', '베이스 메이크업용 메이크업베이스':'메이크업', '베이스 메이크업용 컨실러':'메이크업',
       '베이스 메이크업용 쿠션팩트':'메이크업', '베이스 메이크업용 트윈케이크':'메이크업', '베이스 메이크업용 파우더팩트':'메이크업',
       '베이스 메이크업용 파운데이션':'메이크업', '베이스 메이크업용 프라이머':'메이크업', '뷰티 눈썹정리도구':'뷰티소품', '뷰티 속눈썹/쌍꺼풀':'뷰티소품',
       '뷰티 손거울':'뷰티소품', '뷰티 타투':'뷰티소품', '뷰티 헤어캡':'뷰티소품', '뷰티 화장솜':'뷰티소품', 
       '뷰티 화장품 공병/케이스':'뷰티소품', '뷰티용 기름종이':'뷰티소품',
       '뷰티용 면봉/귀이개':'뷰티소품', '뷰티용 뷰러':'뷰티소품', '뷰티용 샤프너':'뷰티소품', 
       '뷰티용 여드름압출기':'뷰티소품', '색조 메이크업 립글로스':'메이크업',
       '색조 메이크업 립라이너':'메이크업', '색조 메이크업 립밤':'메이크업', 
       '색조 메이크업 립스틱':'메이크업', '색조 메이크업 립틴트':'메이크업',
       '색조 메이크업 마스카라':'메이크업', '색조 메이크업 볼터치':'메이크업', '색조 메이크업 속눈썹영양제':'메이크업',
       '색조 메이크업 쉐딩/하이라이터':'메이크업', '색조 메이크업 아이라이너':'메이크업', 
       '색조 메이크업 아이브로우':'메이크업',
       '색조 메이크업 아이섀도우':'메이크업', '샤워코롱':'바디', '샴푸':'헤어', '선로션':'스킨케어', 
       '선스프레이':'스킨케어', '선케어용 선밤':'스킨케어', '선크림':'스킨케어',
       '선파우더':'메이크업', '세안도구':'뷰티소품', '손톱정리도구':'뷰티소품', 
       '스크럽/필링크림':'클렌징', '스킨케어 곡물팩':'스킨케어', '스킨케어 마스크팩':'스킨케어',
       '스킨케어 수면팩':'스킨케어', '스킨케어 시트마스크팩':'스킨케어', '스킨케어 워시오프팩':'스킨케어', 
       '스킨케어 코팩':'스킨케어', '스킨케어 필오프팩':'스킨케어',
       '애프터선':'스킨케어', '입욕제':'바디', '클렌징 로션':'클렌징', '클렌징 오일':'클렌징', 
       '클렌징 워터/젤':'클렌징', '클렌징 크림':'클렌징', '클렌징 티슈':'클렌징',
       '클렌징 폼':'클렌징', '태닝용 선크림':'스킨케어', '트리트먼트':'헤어', '팩도구':'뷰티소품', 
       '풋스프레이':'바디', '풋워시/스크럽':'바디', '풋크림':'바디',
       '풋패치':'바디', '핸드워시/스크럽':'클렌징', '핸드크림':'바디', '헤어 브러쉬':'뷰티소품', '헤어매니큐어':'네일', '헤어무스':'헤어',
       '헤어스타일링용 염색약':'헤어', '헤어스타일링용 펌제':'헤어', '헤어스타일링용 흑채':'헤어', '헤어스프레이':'헤어', '헤어에센스':'헤어',
       '헤어왁스':'헤어', '헤어젤':'헤어', '헤어케어세트':'헤어', '화장 비누':'클렌징', '화장 퍼프':'뷰티소품', '네일아트':'네일', '향수세트':'향수',
       '기능성 화이트닝 화장품':'기능성 화장품', '색조 메이크업 세트':'메이크업', '여성향수':'향수', '남성향수':'향수'}

In [ ]:
food_buy['mid'] = food_buy['small'].map(array)
food_sns['mid'] = food_sns['small'].map(array)

In [ ]:
beauty_buy['mid'] = beauty_buy['small'].map(array_beauty)
beauty_sns['mid'] = beauty_sns['small'].map(array_beauty)

In [ ]:
home_buy['mid'] = home_buy['small'].map(array2)
home_sns['mid'] = home_sns['small'].map(array2)

## 날씨 데이터

In [ ]:
weather2018 = pd.read_csv(drive_path + 'table2018.txt')
weather2019 = pd.read_csv(drive_path + 'table2019.txt')

In [ ]:
weather2018['tma'] = pd.to_datetime(weather2018['tma']) # 날짜 맞추기
# 필요없는 열 제거
weather2018.drop(columns=['Unnamed: 0','stn_id','min_rhm', 'avg_tca','avg_lmac', 'max_ca', 'avg_ts','sum_fog_dur', 'avg_pa', 'avg_td'], axis=1, inplace=True)
weather2018.sort_values(by='tma', axis=0, inplace=True) # 날짜별로 정렬
weather2018.reset_index(drop=True, inplace=True) # 인덱스 정렬
weather2018.fillna(0, inplace=True) # 결측값 채우기

weather2019['tma'] = pd.to_datetime(weather2019['tma']) # 날짜 맞추기
# 필요없는 열 제거
weather2019.drop(columns=['Unnamed: 0','stn_id','min_rhm', 'avg_tca','avg_lmac', 'max_ca', 'avg_ts','sum_fog_dur', 'avg_pa', 'avg_td'], axis=1, inplace=True)
weather2019.sort_values(by='tma', axis=0, inplace=True) # 날짜별로 정렬
weather2019.reset_index(drop=True, inplace=True) # 인덱스 정렬
weather2019.fillna(0, inplace=True) # 결측값 채우기

In [ ]:
# 눈이나 비가 올 경우 체크
weather2018['check'] = [1 if ((x > 0) or (y > 0)) else 0 for x, y in zip(weather2018['sum_rn'], weather2018['dd_mes'])]

# 눈이나 비가 올 경우 체크
weather2019['check'] = [1 if ((x > 0) or (y > 0)) else 0 for x, y in zip(weather2019['sum_rn'], weather2019['dd_mes'])]

In [ ]:
weather2018.columns = weather2018.columns.str.replace('tma','date')
weather2019.columns = weather2019.columns.str.replace('tma','date')

In [ ]:
result = pd.DataFrame()
result2 = pd.DataFrame()

for i in range(1,7):
    df = pd.read_excel(drive_path + 'period{}.xlsx'.format(i))
    df.columns = df.iloc[2].values
    df = df[df.측정소명 == "평균"].sort_values(by='날짜')
    result = pd.concat([result, df])

for i in range(7, 13):
    df = pd.read_excel(drive_path + 'period{}.xlsx'.format(i))
    df.columns = df.iloc[2].values
    df = df[df.측정소명 == "평균"].sort_values(by='날짜')
    result2 = pd.concat([result2, df])

result[result.날짜 == "전체"]
result.drop(3,axis=0, inplace=True)
result.drop('측정소명', axis=1, inplace=True)
result.columns = ["date", "미세먼지", "초미세먼지", "오존", "이산화질소", "일산화탄소", "아황산가스"]
result['date'] = pd.to_datetime(result['date'])

result2[result2.날짜 == "전체"]
result2.drop(3,axis=0, inplace=True)
result2.drop('측정소명', axis=1, inplace=True)
result2.columns = ["date", "미세먼지", "초미세먼지", "오존", "이산화질소", "일산화탄소", "아황산가스"]
result2['date'] = pd.to_datetime(result2['date'])

weather2018 = pd.merge(weather2018, result, on='date')
weather2019 = pd.merge(weather2019, result2, on='date')

## 새로운 데이터 저장

In [ ]:
beauty_buy.to_csv('beauty_buy.csv', encoding='cp949')
food_buy.to_csv('food_buy.csv', encoding='cp949')
home_buy.to_csv('home_buy.csv', encoding='cp949')

beauty_sns.to_csv('beauty_sns.csv', encoding='cp949')
food_sns.to_csv('food_sns.csv', encoding='cp949')
home_sns.to_csv('home_sns.csv', encoding='cp949')

In [ ]:
weather = pd.concat([weather2018, weather2019], ignore_index=True)

In [ ]:
weather.to_csv('weather.csv', encoding='cp949')